In [ ]:
import pandas as pd
from pathlib import Path
from data_profiling import ProfileReport
import ipywidgets as widgets
from IPython.display import display


# Caminhos
DATA_PATH = Path("../data/raw")
REPORT_PATH = Path("../reports")

# Garante que reports existe
REPORT_PATH.mkdir(exist_ok=True)


# Lista CSVs
datasets = {
    file.name: file 
    for file in DATA_PATH.glob("*.csv")
}


# Menu
menu = widgets.Dropdown(
    options=list(datasets.keys()),
    description="Base:",
    layout=widgets.Layout(width="400px")
)


botao = widgets.Button(
    description="Gerar Relatório",
    button_style="success"
)


saida = widgets.Output()


def gerar_relatorio(b):

    with saida:
        saida.clear_output()

        arquivo = datasets[menu.value]

        print(f"Analisando: {arquivo.name}")

        df = pd.read_csv(arquivo)

        print(f"Linhas: {df.shape[0]}")
        print(f"Colunas: {df.shape[1]}")


        profile = ProfileReport(
            df,
            title=f"FG Profiling - {menu.value}",
            explorative=True
        )


        # Salvar relatório
        nome_relatorio = menu.value.replace(".csv", "_profiling.html")

        caminho_saida = REPORT_PATH / nome_relatorio

        profile.to_file(caminho_saida)

        print(f"\nRelatório salvo em:")
        print(caminho_saida)


        display(profile)


botao.on_click(gerar_relatorio)


display(menu)
display(botao)
display(saida)

Dropdown(description='Base:', layout=Layout(width='400px'), options=('olist_customers_dataset.csv', 'olist_geo…

Button(button_style='success', description='Gerar Relatório', style=ButtonStyle())

Output()

In [6]:
import pandas as pd
from pathlib import Path
import pandera.pandas as pa
import ipywidgets as widgets
from IPython.display import display


# Caminhos
DATA_PATH = Path("../data/raw")


# Lista CSVs
datasets = {
    file.name: file
    for file in DATA_PATH.glob("*.csv")
}


# Menu
menu = widgets.Dropdown(
    options=list(datasets.keys()),
    description="Base:",
    layout=widgets.Layout(width="400px")
)


botao = widgets.Button(
    description="Validar Base",
    button_style="success"
)


saida = widgets.Output()


def validar_base(b):

    with saida:
        saida.clear_output()

        arquivo = datasets[menu.value]

        print(f"Validando: {arquivo.name}")

        df = pd.read_csv(arquivo)

        print(f"Linhas: {df.shape[0]}")
        print(f"Colunas: {df.shape[1]}")
        print()


        # Infere um schema inicial a partir do dataframe
        schema = pa.infer_schema(df)


        print("SCHEMA INFERIDO")
        print("-" * 50)

        for coluna, regra in schema.columns.items():

            print(
                f"{coluna:<40} "
                f"dtype={str(regra.dtype):<15} "
                f"nullable={regra.nullable}"
            )


        print("\nVALIDAÇÃO")
        print("-" * 50)


        try:

            schema.validate(
                df,
                lazy=True
            )

            print("✅ Base validada com sucesso.")
            print("Nenhum erro encontrado pelo schema.")


        except pa.errors.SchemaErrors as erro:

            print("❌ Foram encontrados problemas na base.\n")

            erros = erro.failure_cases

            display(erros)


botao.on_click(validar_base)


display(menu)
display(botao)
display(saida)

Dropdown(description='Base:', layout=Layout(width='400px'), options=('olist_customers_dataset.csv', 'olist_geo…

Button(button_style='success', description='Validar Base', style=ButtonStyle())

Output()

In [10]:
import pandas as pd
from pathlib import Path
import pandera.pandas as pa
from IPython.display import display


# Caminho da base
DATA_PATH = Path("../data/raw/olist_orders_dataset.csv")


# Leitura
df = pd.read_csv(DATA_PATH)


print("ESTUDO DA BASE COM PANDERA")
print("=" * 60)

print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")

print("\nCOLUNAS")
print("-" * 60)

for coluna in df.columns:
    print(coluna)


# ------------------------------------------------------
# Inferência do schema
# ------------------------------------------------------

schema = pa.infer_schema(df)


print("\nSCHEMA INFERIDO")
print("-" * 60)

resultado = []

for coluna, config in schema.columns.items():

    resultado.append({
        "coluna": coluna,
        "dtype_pandera": str(config.dtype),
        "nullable": config.nullable,
        "unique": config.unique,
        "required": config.required
    })


schema_df = pd.DataFrame(resultado)

display(schema_df)


# ------------------------------------------------------
# Informações complementares da base
# ------------------------------------------------------

print("\nNULOS")
print("-" * 60)

nulos = pd.DataFrame({
    "coluna": df.columns,
    "qtd_nulos": df.isna().sum().values,
    "percentual_nulos": (
        df.isna().mean().values * 100
    ).round(2)
})

display(nulos)


print("\nCARDINALIDADE")
print("-" * 60)

cardinalidade = pd.DataFrame({
    "coluna": df.columns,
    "valores_unicos": [
        df[col].nunique(dropna=True)
        for col in df.columns
    ]
})

display(cardinalidade)


print("\nTIPOS IDENTIFICADOS PELO PANDAS")
print("-" * 60)

tipos = pd.DataFrame({
    "coluna": df.columns,
    "dtype": df.dtypes.astype(str).values
})

display(tipos)


print("\nSCHEMA PANDERA COMPLETO")
print("-" * 60)

print(schema)

ESTUDO DA BASE COM PANDERA
Linhas: 99441
Colunas: 8

COLUNAS
------------------------------------------------------------
order_id
customer_id
order_status
order_purchase_timestamp
order_approved_at
order_delivered_carrier_date
order_delivered_customer_date
order_estimated_delivery_date

SCHEMA INFERIDO
------------------------------------------------------------


,coluna,dtype_pandera,nullable,unique,required
0,order_id,object,False,False,True
1,customer_id,object,False,False,True
2,order_status,object,False,False,True
3,order_purchase_timestamp,object,False,False,True
4,order_approved_at,object,True,False,True
5,order_delivered_carrier_date,object,True,False,True
6,order_delivered_customer_date,object,True,False,True
7,order_estimated_delivery_date,object,False,False,True



NULOS
------------------------------------------------------------


,coluna,qtd_nulos,percentual_nulos
0,order_id,0,0.00
1,customer_id,0,0.00
2,order_status,0,0.00
3,order_purchase_timestamp,0,0.00
4,order_approved_at,160,0.16
5,order_delivered_carrier_date,1783,1.79
6,order_delivered_customer_date,2965,2.98
7,order_estimated_delivery_date,0,0.00



CARDINALIDADE
------------------------------------------------------------


,coluna,valores_unicos
0,order_id,99441
1,customer_id,99441
2,order_status,8
3,order_purchase_timestamp,98875
4,order_approved_at,90733
5,order_delivered_carrier_date,81018
6,order_delivered_customer_date,95664
7,order_estimated_delivery_date,459



TIPOS IDENTIFICADOS PELO PANDAS
------------------------------------------------------------


,coluna,dtype
0,order_id,object
1,customer_id,object
2,order_status,object
3,order_purchase_timestamp,object
4,order_approved_at,object
5,order_delivered_carrier_date,object
6,order_delivered_customer_date,object
7,order_estimated_delivery_date,object



SCHEMA PANDERA COMPLETO
------------------------------------------------------------
<Schema DataFrameSchema(
    columns={
        'order_id': <Schema Column(name=order_id, type=DataType(object))>
        'customer_id': <Schema Column(name=customer_id, type=DataType(object))>
        'order_status': <Schema Column(name=order_status, type=DataType(object))>
        'order_purchase_timestamp': <Schema Column(name=order_purchase_timestamp, type=DataType(object))>
        'order_approved_at': <Schema Column(name=order_approved_at, type=DataType(object))>
        'order_delivered_carrier_date': <Schema Column(name=order_delivered_carrier_date, type=DataType(object))>
        'order_delivered_customer_date': <Schema Column(name=order_delivered_customer_date, type=DataType(object))>
        'order_estimated_delivery_date': <Schema Column(name=order_estimated_delivery_date, type=DataType(object))>
    },
    checks=[],
    parsers=[],
    coerce=True,
    dtype=None,
    index=<Schema Index(